In [1]:
import sys
sys.path.append("../src")
from data_prep import process_subject_session

windows, labels = process_subject_session(
    "../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf",
    "../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_channels.tsv"
)

print("windows shape:", windows.shape)
print("labels shape:", labels.shape)
print("movement fraction:", labels.mean())
print("channels kept:", windows.shape[1])

windows shape: (2185, 19, 50)
labels shape: (2185,)
movement fraction: 0.16384439359267736
channels kept: 19


In [2]:
import glob

edf_files = sorted(glob.glob("../data/ds005931/sub-01/ses-*/ieeg/*_ieeg.edf"))
print(edf_files)

['../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf', '../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf']


In [3]:
from data_prep import process_subject_session

all_windows = []
all_labels = []

for edf_path in edf_files:
    channels_path = edf_path.replace("_ieeg.edf", "_channels.tsv")
    print(f"Processing {edf_path}...")
    w, l = process_subject_session(edf_path, channels_path)
    print(f"  -> {w.shape[0]} windows, {w.shape[1]} channels, movement frac {l.mean():.3f}")
    all_windows.append(w)
    all_labels.append(l)

print("Channel counts per session:", [w.shape[1] for w in all_windows])

Processing ../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf...
  -> 2185 windows, 19 channels, movement frac 0.164
Processing ../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf...
  -> 2185 windows, 64 channels, movement frac 0.164
Channel counts per session: [19, 64]


In [4]:
import importlib
import data_prep
importlib.reload(data_prep)   # picks up the new functions without restarting the kernel
from data_prep import get_common_good_channels, process_subject_session_fixed

session_paths = [(p, p.replace("_ieeg.edf", "_channels.tsv")) for p in edf_files]
common_channels = get_common_good_channels(session_paths)
print(f"Common good channels across sessions: {len(common_channels)}")
print(sorted(common_channels))

Common good channels across sessions: 0
[]


In [7]:
for edf_path, channels_path in session_paths:
    good = load_good_channels(channels_path)
    raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
    name_map = {clean_channel_name(ch): ch for ch in raw.ch_names}
    good_in_edf = {n for n in good if n in name_map}
    print(edf_path)
    print(f"  good (from tsv): {len(good)}  -> {sorted(good)[:10]}...")
    print(f"  good AND in EDF: {len(good_in_edf)} -> {sorted(good_in_edf)[:10]}...")
    print(f"  all EDF channel names (cleaned): {sorted(name_map.keys())[:10]}...")

../data/ds005931/sub-01/ses-01/ieeg/sub-01_ses-01_task-game_run-01_ieeg.edf
  good (from tsv): 83  -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  good AND in EDF: 19 -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  all EDF channel names (cleaned): ['A1', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18']...
../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf
  good (from tsv): 83  -> ['A2', 'A27', 'A28', 'A29', 'A3', 'A37', 'A38', 'A39', 'A4', 'A40']...
  good AND in EDF: 64 -> ['B1', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18']...
  all EDF channel names (cleaned): ['B1', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18']...


In [6]:
from data_prep import (
    process_subject_session,
    get_common_good_channels,
    process_subject_session_fixed,
    load_good_channels,
    clean_channel_name,
)
import mne

In [8]:
from data_prep import process_subject_session

edf_path = "../data/ds005931/sub-01/ses-02/ieeg/sub-01_ses-02_task-game_run-01_ieeg.edf"
channels_path = edf_path.replace("_ieeg.edf", "_channels.tsv")

windows, labels = process_subject_session(edf_path, channels_path)
print("windows shape:", windows.shape)
print("labels shape:", labels.shape)
print("movement fraction:", labels.mean())

windows shape: (2185, 64, 50)
labels shape: (2185,)
movement fraction: 0.16384439359267736


In [9]:
import numpy as np

np.savez("../data/processed_sub01_ses02.npz", windows=windows, labels=labels)
print("Saved.")

Saved.
